In [0]:
df_categoria = spark.read.format("delta").table("silver.categoria")
df_marca     = spark.read.format("delta").table("silver.marca")
df_modelo    = spark.read.format("delta").table("silver.modelo")
df_estado    = spark.read.format("delta").table("silver.estado")
df_cidade    = spark.read.format("delta").table("silver.cidade")
df_agencia   = spark.read.format("delta").table("silver.agencia")
df_cliente   = spark.read.format("delta").table("silver.cliente")
df_carro     = spark.read.format("delta").table("silver.carro")
df_reserva   = spark.read.format("delta").table("silver.reserva")
df_pagamento = spark.read.format("delta").table("silver.pagamento")

Remove as tabelas antigas antes de recriar

In [0]:
%sql
DROP TABLE IF EXISTS gold.fato_reserva;
DROP TABLE IF EXISTS gold.dim_cliente;
DROP TABLE IF EXISTS gold.dim_carro;
DROP TABLE IF EXISTS gold.dim_localidade;
DROP TABLE IF EXISTS gold.dim_tempo;

Criar tabelas dimensionais e fato:(Ralph Kimball)

In [0]:
%sql

-- Dimensão Cliente
CREATE TABLE IF NOT EXISTS gold.dim_cliente (
    SK_CLIENTE          BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_CLIENTE      INT,
    NOME_CLIENTE        VARCHAR(100),
    CPF                 VARCHAR(11),
    SEXO                CHAR(1),
    DATA_NASCIMENTO     DATE
) USING DELTA;

-- Dimensão Carro
CREATE TABLE IF NOT EXISTS gold.dim_carro (
    SK_CARRO            BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_CARRO        INT,
    NOME_MARCA          VARCHAR(100),
    NOME_MODELO         VARCHAR(100),
    NOME_CATEGORIA      VARCHAR(50),
    ANO                 INT,
    COR                 VARCHAR(30),
    PLACA               VARCHAR(10)
) USING DELTA;

-- Dimensão Localidade
CREATE TABLE IF NOT EXISTS gold.dim_localidade (
    SK_LOCALIDADE       BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_CIDADE       INT,
    NOME_CIDADE         VARCHAR(100),
    NOME_ESTADO         VARCHAR(100),
    SIGLA_UF            CHAR(2)
) USING DELTA;

-- Dimensão Tempo
CREATE TABLE IF NOT EXISTS gold.dim_tempo (
    DATA                DATE,
    ANO                 INT,
    MES                 INT,
    NOME_MES            VARCHAR(20),
    DIA                 INT,
    NOME_DIA_SEMANA     VARCHAR(20)
) USING DELTA;

-- Tabela Fato
CREATE TABLE IF NOT EXISTS gold.fato_reserva (
    FK_TEMPO            DATE,
    FK_CLIENTE          BIGINT,
    FK_CARRO            BIGINT,
    FK_LOCALIDADE       BIGINT,
    VALOR_TOTAL         NUMERIC(10,2),
    VALOR_DIARIA        NUMERIC(10,2),
    QTDE_DIAS           INT,
    STATUS_RESERVA      VARCHAR(20)
) USING DELTA;

Criar views temporárias:

In [0]:
df_categoria.createOrReplaceTempView("categoria")
df_marca.createOrReplaceTempView("marca")
df_modelo.createOrReplaceTempView("modelo")
df_estado.createOrReplaceTempView("estado")
df_cidade.createOrReplaceTempView("cidade")
df_agencia.createOrReplaceTempView("agencia")
df_cliente.createOrReplaceTempView("cliente")
df_carro.createOrReplaceTempView("carro")
df_reserva.createOrReplaceTempView("reserva")
df_pagamento.createOrReplaceTempView("pagamento")

Popular dim_cliente:

In [0]:
%sql
MERGE INTO gold.dim_cliente AS d
USING (
    SELECT CODIGO_CLIENTE, NOME_CLIENTE, CPF, SEXO, DATA_NASCIMENTO FROM cliente
) AS s
ON d.CODIGO_CLIENTE = s.CODIGO_CLIENTE
WHEN MATCHED AND (d.NOME_CLIENTE <> s.NOME_CLIENTE OR d.CPF <> s.CPF) THEN
    UPDATE SET NOME_CLIENTE = s.NOME_CLIENTE, CPF = s.CPF, SEXO = s.SEXO, DATA_NASCIMENTO = s.DATA_NASCIMENTO
WHEN NOT MATCHED THEN
    INSERT (CODIGO_CLIENTE, NOME_CLIENTE, CPF, SEXO, DATA_NASCIMENTO)
    VALUES (s.CODIGO_CLIENTE, s.NOME_CLIENTE, s.CPF, s.SEXO, s.DATA_NASCIMENTO)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
18,0,0,18


Popular dim_carro:

In [0]:
%sql
MERGE INTO gold.dim_carro AS d
USING (
    SELECT c.CODIGO_CARRO, ma.NOME_MARCA, mo.NOME_MODELO, cat.NOME_CATEGORIA, c.ANO, c.COR, c.PLACA
    FROM carro c
    INNER JOIN modelo mo ON c.CODIGO_MODELO = mo.CODIGO_MODELO
    INNER JOIN marca ma ON mo.CODIGO_MARCA = ma.CODIGO_MARCA
    INNER JOIN categoria cat ON mo.CODIGO_CATEGORIA = cat.CODIGO_CATEGORIA
) AS s
ON d.CODIGO_CARRO = s.CODIGO_CARRO
WHEN MATCHED AND (d.NOME_MARCA <> s.NOME_MARCA OR d.NOME_MODELO <> s.NOME_MODELO) THEN
    UPDATE SET NOME_MARCA = s.NOME_MARCA, NOME_MODELO = s.NOME_MODELO, NOME_CATEGORIA = s.NOME_CATEGORIA,
               ANO = s.ANO, COR = s.COR, PLACA = s.PLACA
WHEN NOT MATCHED THEN
    INSERT (CODIGO_CARRO, NOME_MARCA, NOME_MODELO, NOME_CATEGORIA, ANO, COR, PLACA)
    VALUES (s.CODIGO_CARRO, s.NOME_MARCA, s.NOME_MODELO, s.NOME_CATEGORIA, s.ANO, s.COR, s.PLACA)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
15,0,0,15


Popular dim_localidade: cidade + estado

In [0]:
%sql
MERGE INTO gold.dim_localidade AS d
USING (
    SELECT c.CODIGO_CIDADE, c.NOME_CIDADE, e.NOME_ESTADO, e.SIGLA_UF
    FROM cidade c
    INNER JOIN estado e ON c.CODIGO_ESTADO = e.CODIGO_ESTADO
) AS s
ON d.CODIGO_CIDADE = s.CODIGO_CIDADE
WHEN MATCHED AND (d.NOME_CIDADE <> s.NOME_CIDADE OR d.NOME_ESTADO <> s.NOME_ESTADO) THEN
    UPDATE SET NOME_CIDADE = s.NOME_CIDADE, NOME_ESTADO = s.NOME_ESTADO, SIGLA_UF = s.SIGLA_UF
WHEN NOT MATCHED THEN
    INSERT (CODIGO_CIDADE, NOME_CIDADE, NOME_ESTADO, SIGLA_UF)
    VALUES (s.CODIGO_CIDADE, s.NOME_CIDADE, s.NOME_ESTADO, s.SIGLA_UF)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,0,0,12


Popular dim_tempo:

In [0]:
from pyspark.sql.functions import expr

data_inicial = "2023-01-01"
data_final   = "2026-12-31"

num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

df_tempo = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS DATA") \
    .selectExpr(
        "DATA",
        "year(DATA) AS ANO",
        "month(DATA) AS MES",
        """(CASE month(DATA)
            WHEN 1 THEN 'JANEIRO' WHEN 2 THEN 'FEVEREIRO' WHEN 3 THEN 'MARCO'
            WHEN 4 THEN 'ABRIL' WHEN 5 THEN 'MAIO' WHEN 6 THEN 'JUNHO'
            WHEN 7 THEN 'JULHO' WHEN 8 THEN 'AGOSTO' WHEN 9 THEN 'SETEMBRO'
            WHEN 10 THEN 'OUTUBRO' WHEN 11 THEN 'NOVEMBRO' WHEN 12 THEN 'DEZEMBRO'
        END) AS NOME_MES""",
        "day(DATA) AS DIA",
        """(CASE dayofweek(DATA)
            WHEN 1 THEN 'DOMINGO' WHEN 2 THEN 'SEGUNDA-FEIRA' WHEN 3 THEN 'TERCA-FEIRA'
            WHEN 4 THEN 'QUARTA-FEIRA' WHEN 5 THEN 'QUINTA-FEIRA'
            WHEN 6 THEN 'SEXTA-FEIRA' WHEN 7 THEN 'SABADO'
        END) AS NOME_DIA_SEMANA"""
    )

df_tempo.write.mode("overwrite").saveAsTable("gold.dim_tempo", format="delta")

Popular fato_reserva:juntando reserva com todas as dimensões

In [0]:
%sql
INSERT INTO gold.fato_reserva
SELECT
    r.DATA_RETIRADA,
    dc.SK_CLIENTE,
    dcar.SK_CARRO,
    dloc.SK_LOCALIDADE,
    r.VALOR_TOTAL,
    r.VALOR_DIARIA,
    DATEDIFF(r.DATA_DEVOLUCAO, r.DATA_RETIRADA) AS QTDE_DIAS,
    r.STATUS_RESERVA
FROM reserva r
    INNER JOIN gold.dim_cliente dc   ON r.CODIGO_CLIENTE = dc.CODIGO_CLIENTE
    INNER JOIN carro c               ON r.CODIGO_CARRO = c.CODIGO_CARRO
    INNER JOIN gold.dim_carro dcar   ON c.CODIGO_CARRO = dcar.CODIGO_CARRO
    INNER JOIN agencia a             ON r.CODIGO_AGENCIA = a.CODIGO_AGENCIA
    INNER JOIN gold.dim_localidade dloc ON a.CODIGO_CIDADE = dloc.CODIGO_CIDADE
    INNER JOIN gold.dim_tempo dt     ON r.DATA_RETIRADA = dt.DATA

num_affected_rows,num_inserted_rows
40,40


In [0]:
%sql
SELECT * FROM gold.dim_cliente

SK_CLIENTE,CODIGO_CLIENTE,NOME_CLIENTE,CPF,SEXO,DATA_NASCIMENTO
1,1,Carlos Eduardo Souza,4823917562,M,1985-03-12
2,2,Mariana Lima Ferreira,31276540089,F,1992-07-25
3,3,Roberto Alves Pereira,72384910036,M,1978-11-08
4,4,Fernanda Costa Oliveira,58103274910,F,1990-01-30
5,5,Lucas Henrique Martins,29471038540,M,1995-06-14
6,6,Juliana Ramos Nascimento,83056192740,F,1988-09-03
7,7,André Luís Carvalho,61790243851,M,1982-04-22
8,8,Patrícia Souza Barbosa,17432896005,F,1997-12-17
9,9,Thiago Mendes Ribeiro,94025163720,M,1980-08-05
10,10,Camila Aparecida Santos,38617250094,F,1993-02-28


In [0]:
%sql
SELECT * FROM gold.dim_carro

SK_CARRO,CODIGO_CARRO,NOME_MARCA,NOME_MODELO,NOME_CATEGORIA,ANO,COR,PLACA
1,1,Fiat,Argo,Econômico,2022,Branco,BRA2E34
2,3,Volkswagen,Polo,Econômico,2021,Preto,DEF3J56
3,2,Chevrolet,Onix,Econômico,2023,Prata,GHI5K78
4,9,Renault,Kwid,Econômico,2021,Branco,YZA2S90
5,5,Chevrolet,Cruze,Intermediário,2022,Cinza,MNO7N12
6,10,Volkswagen,Jetta,Intermediário,2022,Prata,BCD3T12
7,7,Toyota,Corolla Cross,SUV,2023,Vermelho,STU9Q56
8,4,Hyundai,Creta,SUV,2023,Azul,JKL6M90
9,11,Toyota,SW4,SUV,2023,Preto,EFG4U34
10,8,Volkswagen,Tiguan,SUV,2022,Cinza,VWX1R78


In [0]:
%sql
SELECT * FROM gold.dim_localidade

SK_LOCALIDADE,CODIGO_CIDADE,NOME_CIDADE,NOME_ESTADO,SIGLA_UF
1,1,São Paulo,São Paulo,SP
2,2,Campinas,São Paulo,SP
3,3,Santos,São Paulo,SP
4,4,Rio de Janeiro,Rio de Janeiro,RJ
5,5,Niterói,Rio de Janeiro,RJ
6,6,Belo Horizonte,Minas Gerais,MG
7,7,Uberlândia,Minas Gerais,MG
8,8,Curitiba,Paraná,PR
9,9,Londrina,Paraná,PR
10,10,Porto Alegre,Rio Grande do Sul,RS


In [0]:
%sql
SELECT * FROM gold.dim_tempo

DATA,ANO,MES,NOME_MES,DIA,NOME_DIA_SEMANA
2026-07-02,2026,7,JULHO,2,QUINTA-FEIRA
2026-07-03,2026,7,JULHO,3,SEXTA-FEIRA
2026-07-04,2026,7,JULHO,4,SABADO
2026-07-05,2026,7,JULHO,5,DOMINGO
2026-07-06,2026,7,JULHO,6,SEGUNDA-FEIRA
2026-07-07,2026,7,JULHO,7,TERCA-FEIRA
2026-07-08,2026,7,JULHO,8,QUARTA-FEIRA
2026-07-09,2026,7,JULHO,9,QUINTA-FEIRA
2026-07-10,2026,7,JULHO,10,SEXTA-FEIRA
2026-07-11,2026,7,JULHO,11,SABADO


In [0]:
%sql
SELECT * FROM gold.fato_reserva

FK_TEMPO,FK_CLIENTE,FK_CARRO,FK_LOCALIDADE,VALOR_TOTAL,VALOR_DIARIA,QTDE_DIAS,STATUS_RESERVA
2023-02-10,1,1,1,600.00,120.00,5,concluida
2023-03-05,2,2,4,450.00,150.00,3,concluida
2023-04-20,3,5,6,1400.00,200.00,7,concluida
2023-05-01,4,7,8,880.00,220.00,4,concluida
2023-06-15,5,4,10,330.00,110.00,3,concluida
2023-07-22,6,3,1,875.00,125.00,7,concluida
2023-08-10,7,8,4,920.00,230.00,4,concluida
2023-09-03,8,11,6,2450.00,350.00,7,concluida
2023-10-18,9,10,8,960.00,240.00,4,concluida
2023-11-05,10,6,10,620.00,155.00,4,concluida


Supabase (banco real)

    ↓ notebook 002 (extração via JDBC)

Landing (CSVs no Volume)
    
    ↓ notebook 003 (Bronze)

Tabelas Delta Lake Bronze

    ↓ notebook 004 (Silver - Data Quality)

Tabelas Delta Lake Silver

    ↓ notebook 005 (Gold - MERGE INTO)

Tabelas Dimensionais + Fato 